# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Install the mlcroissant library if not already available
!pip install mlcroissant

## 1. Data Loading

First, load the dataset metadata using `mlcroissant`. This metadata describes the dataset’s record sets, fields, and other properties.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the FAIR² dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Display basic dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print("Version:", getattr(metadata, 'version', 'N/A'))
print("License:", getattr(metadata, 'license', 'N/A'))
print("Temporal coverage:", getattr(metadata, 'temporalCoverage', 'N/A'))
print("Spatial coverage:", getattr(metadata, 'spatialCoverage', 'N/A'))

## 2. Data Overview

Review available record sets and their fields. All references will be made by `@id` for disambiguation and clarity as required by the FAIR² Croissant schema.

This step will enumerate the available record sets, their `@id`s, and the fields within each.

In [ ]:
# List all available record sets and their associated fields, using @id references
print("Available Record Sets in this dataset:")
record_sets = []

for record_set in getattr(metadata, 'recordSet', []):
    print("- @id:", record_set['@id'])
    record_sets.append(record_set['@id'])
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        if isinstance(field, dict):
            print(f"    - @id: {field.get('@id', 'N/A')}  name: {field.get('name', 'N/A')}  dataType: {field.get('dataType', 'N/A')}")
        else:
            print(f"    - {field}")
    print("")
if not record_sets:
    print("No record sets found in the metadata. The dataset may rely on distributions or further nesting.")


### Note
If no record sets are shown above, it means the dataset only includes record sets accessible via its distributions, or via subsequent dynamic fetching by `mlcroissant`. In such cases, data may be loaded directly by exploring the distributions.

In [ ]:
# List distributions (data resources) available in the package, with their @id
print("Distributions (Data resources):")
for dist in getattr(metadata, 'distribution', []):
    print("- @id:", dist['@id'])

## 3. Data Extraction

We'll attempt to extract data using each available record set using their `@id`. If no explicit record sets are in the metadata, we will attempt to extract available record sets dynamically loaded by mlcroissant from the dataset.

In [ ]:
# Helper to list all record sets programmatically
print("Dynamic discovery of record sets...")
found_record_sets = dataset.record_set_ids()
if not found_record_sets:
    print("No record sets found by mlcroissant (the dataset may be documentation only or requires additional configuration).")
else:
    for rid in found_record_sets:
        print("- record set @id:", rid)

# Let's load all found record sets into dataframes
dataframes = {}
for rs_id in found_record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded record set: {rs_id} with {df.shape[0]} rows and columns: {df.columns.tolist()}")
        else:
            print(f"Found record set but no records: {rs_id}")
    except Exception as e:
        print(f"Error extracting record set {rs_id}: {e}")

if dataframes:
    # Print preview of the first dataframe
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nFirst 5 rows of record set {first_rs_id}:")
    display(dataframes[first_rs_id].head())
else:
    print("No tabular records found in any record set.")

## 4. Exploratory Data Analysis (EDA)

In this section we perform analysis on one record set. All operations below use field/column `@id` exclusively. Adjust code according to data actually loaded above.

In [ ]:
# Select the record set @id and field @id you want to work with

# Use first available DataFrame if present
if not dataframes:
    print("No dataframes to analyze; please check earlier extraction steps.")
else:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Identify a likely numeric field (by attempting conversion)
    numeric_field_id = None
    for col in df.columns:
        # Try to cast to float; if possible, probably numeric
        try:
            if pd.to_numeric(df[col], errors='coerce').notnull().sum() > 0:
                numeric_field_id = col
                break
        except Exception:
            continue

    if numeric_field_id is None:
        print("No numeric field found in dataframe.")
    else:
        print(f"Using field '@id': {numeric_field_id} as a numeric field for demonstration.")

        # Convert to numeric for analysis
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        # Filter for values greater than a demonstration threshold (e.g., > 10)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()

        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to group by another field (excluding ourselves and seeking categorical)
        group_field_id = None
        for candidate in df.columns:
            if candidate != numeric_field_id and df[candidate].nunique() < df.shape[0] / 2:
                group_field_id = candidate
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No suitable field found for grouping.")

## 5. Visualization

Visualize the distribution of the selected numeric field and, if a grouping field was found above, compare group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes or numeric_field_id is None:
    print("No numeric field for visualization.")
else:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load and explore the FAIR² dataset on adoption predictors using the `mlcroissant` library and Jupyter. Data was accessed via Croissant schema `@id` references, processed into DataFrames, and visualized to reveal numeric field distributions and categorical groupings. Adjust field and record set references as needed for deeper analyses based on your research questions!